Rodando modelo localmente

In [13]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Portuguese BR. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
print(ai_msg.content)

Eu adoro programar.


Usando tools pela API de chat, nesse caso o usuário e responsável pela execução e chamado

In [ ]:
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import ToolMessage

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

llm = ChatOllama(model="llama3.2:1b")
llm_tool = llm.bind_tools([multiply, add])

messages = [
    (
        "system",
        "You are a helpful assistant that can use the tools to multiply and add numbers. Use the tools when the user asks you to perform these operations.",
    ),
    ("human", "What is 5 multiplied by 3?"),
    ("human", "What is 10 added to 4?")
]

resp = llm_tool.invoke(messages)

# Pra printar todas as mensagens precisou executar 3 vezes o modelo, pq cada tool call é uma nova mensagem pro modelo, 
# e ele precisa processar a resposta da tool pra gerar a próxima resposta.
def get_final_response(messages, resp, resultado, tool_call):
    resp_final = llm_tool.invoke([*messages, resp, ToolMessage(content=str(resultado), tool_call_id=tool_call['id'])])
    print(f"Final response from model: {resp_final.content}")

# Verificar a chamada da ferramenta
if resp.tool_calls:
    for tool_call in resp.tool_calls: # -> Itera sobre as chamadas de ferramentas feitas pelo modelo
        print(f"Tool called: {tool_call['name']} with arguments {tool_call['args']}")
        if tool_call['name'] == "multiply":
            resultado = multiply.invoke(tool_call['args']) # -> Chama a função de multiplicação com os argumentos fornecidos
            print(f"Result of multiplication: {resultado}")
            get_final_response(messages, resp, resultado, tool_call)
        elif tool_call['name'] == "add":
            resultado = add.invoke(tool_call['args']) # -> Chama a função de adição com os argumentos fornecidos
            print(f"Result of addition: {resultado}")
            get_final_response(messages, resp, resultado, tool_call)
            
            # Enviar a resposta de volta para o modelo
            # resp_final = llm_tool.invoke([*messages, resp, ToolMessage(content=str(resultado), tool_call_id=tool_call['id'])])
            # print(f"Final response from model: {resp_final.content}")

# Pq precisa desse processo manual?
# Motivo: O LangChain não pode assumir como você quer executar a tool:
# Tool pode ser uma API paga (precisa de auth)
# Pode exigir tratamento de erro customizado
# Pode precisar logar a execução
# Pode precisar validar dados antes/depois

Tool called: multiply with arguments {'b': '3', 'a': '5'}
Result of multiplication: 15
Final response from model: The result of multiplying 5 by 3 is 15.

The result of adding 10 to 4 is 14.
Tool called: add with arguments {'a': '10', 'b': '4'}
Result of addition: 14
Final response from model: This means that 5 multiplied by 3 equals 15, and 10 added to 4 equals 14.


Usando tool,as com o create_agent

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

messages = [
    (
        "system",
        "You are a helpful assistant that can use the tools to multiply and add numbers. Use the tools when the user asks you to perform these operations.",
    ),
    ("human", "What is 5 multiplied by 3?"),
    ("human", "What is 10 added to 4?")
]

llm = ChatOllama(model="llama3.2:1b")

# Na criação do agente, passamos as ferramentas e o prompt do sistema. 
# O agente vai cuidar de toda a lógica de chamar as ferramentas e processar as respostas.
agente = create_agent(llm, tools=[multiply, add], system_prompt=messages[0][1])

# Chamada de uma só ferramenta
resp = agente.invoke({"messages": messages[1]}) # -> Passamos as mensagens do usuário para o agente processar
print(f"Response from agent: {resp["messages"][-1].content}")

# Chamada de múltiplas ferramentas
resp = agente.invoke({"messages": messages[1:]}) # -> Passamos todas as mensagens do usuário para o agente processar
print(f"Response from agent: {resp["messages"][-1].content}")

Response from agent: The result of multiplying 5 by 3 is 15.

You have requested to format an answer in a specific way. In this case, I've returned the numerical value as it's the most straightforward outcome of the operation. If you'd like me to reformat the output in some other way (e.g., using sentence fragments or combining numbers with text), let me know and I'll be happy to assist!
Response from agent: The multiplication result is: 5 × 3 = 15

The addition result is: 10 + 4 = 14


In [ ]:
# Execução automática da tool
# Código mais simples e completo
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

messages = [
    {"messages": [("user", "Whats is 5 * 3?")]},
    {"messages": [("user", "Whats is 10 + 4?")]}
]

for message in messages:
    agent = create_agent(
        model=ChatOllama(model="llama3.2:1b"),
        system_prompt="You are a calculator",
        tools=[multiply, add]
    )

    # result = agent.invoke(message)
    # print(result["messages"])
    
    # Nem sempre o modelo retorna a resposta final 
    # com esse chamada
    # print(result["messages"][-1].content)

    # Essa é pra tentar corrigir esse problema
    # Mesmo essa não corrigiiu
    print(agent.invoke(message)["messages"][-1].content)

The result of multiplying 5 and 3 is 15.
The result of the addition operation: 10 + 4 = 14.


Saída estruturada

In [ ]:
# Essa é uma das formas de criar uma saída estruturada, usando o PydanticOutputParser

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda
import re

llm = ChatOllama(model="llama3.2:1b", temperature=0)

class Output(BaseModel):
    pais: str = Field(description="The name of the country")
    capital: str = Field(description="The capital city of the country")
    maior_cidade: str = Field(description="The largest city in the country")
    populacao: str = Field(description="The population size of the country")

parser = PydanticOutputParser(pydantic_object=Output)

def fix_population_formatting(populacao_str):
    # Remove pontos e vírgulas do número da população
    # if isinstance(populacao_str, str):
    populacao_limpa = re.sub(r'[.,]', '', populacao_str)
    return str(populacao_limpa)
    # return int(populacao_str)

def processar_resposta_ai(resposta):
    # Aqui você pode adicionar lógica para processar a resposta do modelo, 
    # como corrigir o formato da população ou validar os dados.
    resposta.populacao = fix_population_formatting(resposta.populacao)
    print(f"Resposta processada: {resposta}")
    return resposta

context = ChatPromptTemplate.from_template("""
Você é especialista em geografia.
                                           
Responda a questao sobre o país: {pais}

{format_instructions}
                                                                               
Use EXATAMENTE este formato:
"pais": "nome do país"
"capital": "nome da capital"
"maior_cidade": "nome da maior cidade"
"populacao": "numero_da_populacao"

Não adicione texto antes ou depois da resposta.
                                        
""")

# Observar a sequência de execução: o modelo gera a resposta, o parser tenta parsear, 
# e depois a RunnableLambda processa a resposta do parser.
prompt = context | llm | parser | RunnableLambda(processar_resposta_ai)

ai_msg = prompt.invoke({"pais": "Peru", "format_instructions": parser.get_format_instructions()})
print(ai_msg.model_dump())

Resposta processada: pais='Peru' capital='Lima' maior_cidade='Cusco' populacao='32989000'
{'pais': 'Peru', 'capital': 'Lima', 'maior_cidade': 'Cusco', 'populacao': '32989000'}
